kinetic energy spectra

In [ ]:
import warnings
from functools import partial

import iris
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.colors as mcol

import numpy as np
import paths
import seaborn as sns
from aeolus.calc import (
    last_n_day_mean,
    time_mean,
    vertical_mean,
)
from aeolus.const import init_const
from aeolus.coord import (
    get_cube_rel_days,
    interp_cube_from_height_to_pressure_levels,
    isel,
)
from aeolus.lfric import load_lfric_raw
from aeolus.model import lfric
from aeolus.plot import figsave, add_custom_legend, subplot_label_generator
from common import (
    EXPERIMENTS,
    GROUPS,
    KW_REF_LINE,
    PA_TO_BAR,
    lfric_callback_uniform_height,
)
from ke_spectrum import (
    ke_spectrum_from_uv,
    ke_spectrum_from_cubelist,
    ke_spectrum_from_vrtdiv,
    U_NAME,
    V_NAME,
    VRT_NAME,
    DIV_NAME,
)
from tqdm import tqdm

warnings.filterwarnings("ignore")
iris.FUTURE.date_microseconds = True

print(*EXPERIMENTS.keys(), sep="\n")

In [ ]:
group_key = "shj_dhj_all"
ds_rg = {}
for exp_key in tqdm(GROUPS[group_key].simulations):
    ds_rg[exp_key] = load_lfric_raw(
        sorted(
            (
                paths.data_work
                / "lfric"
                / EXPERIMENTS[exp_key].group
                / EXPERIMENTS[exp_key].label
            ).glob("lfric_diag_regr_gauss*.nc")
        )[-20:],
        callback=partial(
            lfric_callback_uniform_height,
            model_top_height=EXPERIMENTS[exp_key].const.domain_height.data,
        ),
    )
print(exp_key)
print(ds_rg[exp_key])

In [ ]:
spectra = {}
method = "vrtdiv"
for exp_key in tqdm(GROUPS[group_key].simulations):
    if EXPERIMENTS[exp_key].group == "dhj":
        target_pressure = 10_000
    elif EXPERIMENTS[exp_key].group == "shj":
        target_pressure = 67_500
    else:
        # hd209
        target_pressure = 1_000
    u = ds_rg[exp_key].extract_cube(U_NAME)
    v = ds_rg[exp_key].extract_cube(V_NAME)
    vrt = ds_rg[exp_key].extract_cube(VRT_NAME)
    div = ds_rg[exp_key].extract_cube(DIV_NAME)
    pres = (
        ds_rg[exp_key]
        .extract_cube(lfric.pres)
        .interpolate([(lfric.z, u.coord(lfric.z).points)], iris.analysis.Linear())
    )
    if method == "wind":
        u_plev = interp_cube_from_height_to_pressure_levels(u, pres, [target_pressure])
        v_plev = interp_cube_from_height_to_pressure_levels(v, pres, [target_pressure])
        spectra[exp_key] = ke_spectrum_from_uv(
            u_plev,
            v_plev,
            rsphere=EXPERIMENTS[exp_key].const.radius.data.item(),
        )
    else:
        vrt_plev = interp_cube_from_height_to_pressure_levels(
            vrt, pres, [target_pressure]
        )
        div_plev = interp_cube_from_height_to_pressure_levels(
            div, pres, [target_pressure]
        )

        spectra[exp_key] = ke_spectrum_from_vrtdiv(
            vrt_plev,
            div_plev,
            rsphere=EXPERIMENTS[exp_key].const.radius.data.item(),
        )

In [ ]:
_savefig = True
split_panels = True  # SHJ cases on the left, DHJ cases on the right
# idx_lev = 60  # index into level_height
# spec2d = isel(spec, lfric.z, idx_lev)
# spec2d = vertical_mean(spec)
n_days_mean = 200
target_pressures = {"shj": 67_500, "dhj": 10_000}

if split_panels:
    panel_keys = {
        "shj": [
            exp_key
            for exp_key in GROUPS[group_key].simulations
            if EXPERIMENTS[exp_key].group == "shj"
        ],
        "dhj": GROUPS["dhj_all"].simulations,
    }
    fig, axd = plt.subplot_mosaic(
        [list(panel_keys)], figsize=(10, 3), layout="constrained", sharey=False
    )
else:
    panel_keys = {"all": list(GROUPS[group_key].simulations)}
    fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
    axd = {"all": ax}

iletters = subplot_label_generator()
for panel, exp_keys in panel_keys.items():
    ax = axd[panel]
    ax.set_xlim(1, 200)
    ax.set_xticks([1, 10, 100])
    ax.set_xlabel("Wavenumber, $n$")
    if ax is list(axd.values())[0]:
        ax.set_ylabel("Kinetic energy [$m^2$ $s^{-2}$]")
    sns.despine(ax=ax, offset=10)
    # ax.spines["right"].set_visible(False)
    # ax.spines["top"].set_visible(False)
    # ax.grid(color="w")
    # ax.set_facecolor("#EEEEEE")

    for exp_key in exp_keys:
        spectrum = last_n_day_mean(spectra[exp_key], n_days_mean)

        x = spectrum.coord("wavenumber").points
        y = spectrum.data
        ax.loglog(x, y, **EXPERIMENTS[exp_key].kw_plt)

    x_ref = x[1:-1]  # avoid x=0 if present
    y_ref = y[1:-1]
    n = len(x_ref)
    for p, ls, sl, y_mult, label in [
        # (-5 / 3, "--", slice(n // 50, n // 8), 50, "$n^{-5/3}$"),  # small wavenumbers
        (-3, ":", slice(n // 8, n // 4), 4, "$n^{-3}$"),  # large wavenumbers
        # (-4, "-.", slice(n // 2, n), 8, "$n^{-4}$"),  # larger wavenumbers
    ]:
        xs = x_ref[sl]
        ys_data = y_ref[sl]
        x0, y0 = (
            xs[len(xs) // 6],
            ys_data[len(xs) // 6] * y_mult,
        )  # anchor + vertical offset so it doesn't overlap
        ys = y0 * (xs / x0) ** p
        ax.loglog(xs, ys, **{**KW_REF_LINE, "ls": ls})
        ax.annotate(
            label,
            xy=(xs[-1], ys[-1]),
            xytext=(4, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize="small",
        )

    add_custom_legend(
        ax,
        {
            EXPERIMENTS[exp_key].title: {**EXPERIMENTS[exp_key].kw_plt}
            for exp_key in exp_keys
        },
        # title=GROUPS[group_key].title,
        ncol=1,  # len(exp_keys),
        loc="center left",
        frameon=False,
        bbox_to_anchor=(0.0, 0.4),
        # facecolor="#EEEEEE",
        # edgecolor="none",
    )
    if panel == "all":
        pres_label = ", ".join(
            f"{target_pressures[k] * PA_TO_BAR:>5.3f} bar ({k.upper()})"
            for k in target_pressures
        )
    else:
        pres_label = f"{target_pressures[panel] * PA_TO_BAR:>5.3f} bar"
        ax.set_title(f"({next(iletters)}) {panel.upper()}", loc="left", size="small")
    ax.add_artist(
        AnchoredText(
            f"{n_days_mean:.0f}-day mean, {pres_label}",
            loc="lower left",
            prop={"size": "small"},
            frameon=False,
        )
    )
fig.suptitle(f"{GROUPS[group_key].title} | Kinetic Energy Spectra", size="small")

if _savefig:
    figsave(
        fig,
        paths.drafts
        / f"ke_spec_from_{method}_gauss__{group_key}{'_split' if split_panels else ''}",
    )